In [35]:
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, GPT2Model

In [36]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"

TRAIN_SIZE = 10_000
VAL_SIZE = 872

MAX_LENGTH = 128
BATCH_SIZE = 4
NUM_EPOCHS = 1

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

SEED = 42

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: cuda


In [37]:
# ============================================================
# Reproducibility
# ============================================================

torch.manual_seed(SEED)




In [38]:
# ============================================================
# Load SST-2
# ============================================================

dataset = load_dataset("stanfordnlp/sst2")

print(dataset)

# We use:
#
#   train      -> 10,000 examples
#   validation -> official SST-2 validation set
#
# The official test labels are hidden, so we don't use test here.

train_dataset = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(TRAIN_SIZE))
)

val_dataset = dataset["validation"]

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))



DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
Train size: 10000
Validation size: 872


In [39]:
# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 does not have a native padding token.
# For batching, use EOS as PAD.
tokenizer.pad_token = tokenizer.eos_token




In [40]:
# ============================================================
# Dataset preprocessing
# ============================================================

def tokenize_example(example):
    encoded = tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

    encoded["label"] = example["label"]

    return encoded


train_dataset = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

val_dataset = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)



In [41]:
# ============================================================
# Dynamic padding collator
# ============================================================

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]

    encoded = tokenizer.pad(
        {
            "input_ids": input_ids,
        },
        padding=True,
        return_tensors="pt",
    )

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )

    encoded["labels"] = labels

    return encoded


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)


In [42]:
# ============================================================
# Model
# ============================================================

class GPT2SentimentClassifier(nn.Module):

    def __init__(self, model_name, num_labels=2):
        super().__init__()

        # GPT-2 WITHOUT the language-model head.
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        hidden_size = self.gpt2.config.hidden_size

        # Binary classification head:
        #
        # [B, 768] -> [B, 2]
        self.classifier = nn.Linear(
            hidden_size,
            num_labels,
        )


    def forward(self, input_ids, attention_mask, labels=None):

        # ----------------------------------------------------
        # GPT-2
        #
        # last_hidden_state:
        # [B, T, 768]
        # ----------------------------------------------------

        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        # ----------------------------------------------------
        # Find the position of the last real token
        #
        # attention_mask:
        #
        #   1 1 1 1 1 0 0
        #
        # last real token = position 4
        # ----------------------------------------------------

        sequence_lengths = attention_mask.sum(dim=1) - 1

        batch_indices = torch.arange(
            hidden_states.size(0),
            device=hidden_states.device,
        )

        # [B, 768]
        pooled_hidden_states = hidden_states[
            batch_indices,
            sequence_lengths,
        ]

        # ----------------------------------------------------
        # Classification head
        #
        # [B, 768] -> [B, 2]
        # ----------------------------------------------------

        logits = self.classifier(
            pooled_hidden_states
        )


        # ----------------------------------------------------
        # Classification loss
        # ----------------------------------------------------

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
        }




In [43]:
model = GPT2SentimentClassifier(
    MODEL_NAME,
    num_labels=2,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [44]:
# ============================================================
# Freeze GPT-2
# ============================================================

for parameter in model.gpt2.parameters():
    parameter.requires_grad = False


# Only classifier is trainable.
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in trainable_params
)

print()
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_count)



Total parameters: 124441346
Trainable parameters: 1538


In [45]:
# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)



In [46]:
# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(model, data_loader):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in data_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]
        logits = outputs["logits"]

        predictions = logits.argmax(dim=-1)

        total_loss += loss.item() * labels.size(0)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_examples += labels.size(0)

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return avg_loss, accuracy



In [49]:
# ============================================================
# Training
# ============================================================

model.to(device)

for epoch in range(NUM_EPOCHS):

    model.train()

    running_loss = 0.0

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]


        # Backward
        optimizer.zero_grad()

        loss.backward()

        # Update ONLY the classification head
        optimizer.step()

        running_loss += loss.item()

        if (step + 1) % 100 == 0:

            avg_loss = running_loss / 100

            print(
                f"Epoch {epoch + 1} "
                f"Step {step + 1} "
                f"Loss {avg_loss:.4f}"
            )

            running_loss = 0.0

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
    )

    print()
    print(
        f"Epoch {epoch + 1} "
        f"Validation Loss: {val_loss:.4f} "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )
    print()



Epoch 1 Step 100 Loss 0.7636
Epoch 1 Step 200 Loss 0.7307
Epoch 1 Step 300 Loss 0.6994
Epoch 1 Step 400 Loss 0.6487
Epoch 1 Step 500 Loss 0.6574
Epoch 1 Step 600 Loss 0.6545
Epoch 1 Step 700 Loss 0.6630
Epoch 1 Step 800 Loss 0.5971
Epoch 1 Step 900 Loss 0.5922
Epoch 1 Step 1000 Loss 0.6297
Epoch 1 Step 1100 Loss 0.5882
Epoch 1 Step 1200 Loss 0.6115
Epoch 1 Step 1300 Loss 0.5522
Epoch 1 Step 1400 Loss 0.5517
Epoch 1 Step 1500 Loss 0.6114
Epoch 1 Step 1600 Loss 0.5378
Epoch 1 Step 1700 Loss 0.5536
Epoch 1 Step 1800 Loss 0.5748
Epoch 1 Step 1900 Loss 0.5558
Epoch 1 Step 2000 Loss 0.5996
Epoch 1 Step 2100 Loss 0.5722
Epoch 1 Step 2200 Loss 0.5686
Epoch 1 Step 2300 Loss 0.5402
Epoch 1 Step 2400 Loss 0.5441
Epoch 1 Step 2500 Loss 0.5600

Epoch 1 Validation Loss: 0.4749 Validation Accuracy: 0.7856



In [50]:
# ============================================================
# Example predictions
# ============================================================

model.eval()

examples = [
    "This movie was absolutely fantastic.",
    "The movie was boring and disappointing.",
    "I really enjoyed this film.",
    "I wouldn't recommend this movie.",
]

encoded = tokenizer(
    examples,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

encoded = {
    key: value.to(device)
    for key, value in encoded.items()
}

with torch.no_grad():

    outputs = model(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
    )

    probabilities = torch.softmax(
        outputs["logits"],
        dim=-1,
    )

    predictions = probabilities.argmax(dim=-1)

for text, prediction, probability in zip(
    examples,
    predictions,
    probabilities,
):

    label = (
        "positive"
        if prediction.item() == 1
        else "negative"
    )

    print()
    print("Text:", text)
    print("Prediction:", label)
    print("Probabilities:", probability.cpu().tolist())




Text: This movie was absolutely fantastic.
Prediction: positive
Probabilities: [0.12439426779747009, 0.8756057620048523]

Text: The movie was boring and disappointing.
Prediction: negative
Probabilities: [0.8331819176673889, 0.16681812703609467]

Text: I really enjoyed this film.
Prediction: positive
Probabilities: [0.1540893316268921, 0.8459107279777527]

Text: I wouldn't recommend this movie.
Prediction: negative
Probabilities: [0.7852526903152466, 0.21474727988243103]


In [51]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [52]:
!jupyter nbconvert --to html /content/drive/MyDrive/"Colab Notebooks"/sft_sst2_gpt2_E1-1_2logits.ipynb --output-dir=/content/drive/MyDrive/

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/sft_sst2_gpt2_E1-1_2logits.ipynb to html
[NbConvertApp] Writing 338456 bytes to /content/drive/MyDrive/sft_sst2_gpt2_E1-1_2logits.html
